# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 4: Exploratory Data Analysis with SQL

We load the SpaceX launch dataset into a SQLite database and answer ten questions with SQL queries.

In [1]:
import sqlite3
import pandas as pd

pd.set_option('display.max_colwidth', None)

In [2]:
con = sqlite3.connect("my_data1.db")
cur = con.cursor()

df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False, method="multi")
df.shape

(101, 10)

In [3]:
cur.execute("DROP TABLE IF EXISTS SPACEXTABLE")
cur.execute("CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL")
con.commit()
pd.read_sql("SELECT COUNT(*) AS records FROM SPACEXTABLE", con)

,records
0,101


### Task 1: Display the names of the unique launch sites in the space mission

In [4]:
query = """
SELECT DISTINCT Launch_Site
FROM SPACEXTABLE
"""
pd.read_sql(query, con)

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### Task 2: Display 5 records where launch sites begin with the string 'CCA' 

In [5]:
query = """
SELECT *
FROM SPACEXTABLE
WHERE Launch_Site LIKE 'CCA%'
LIMIT 5
"""
pd.read_sql(query, con)

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Task 3: Display the total payload mass carried by boosters launched by NASA (CRS)

In [6]:
query = """
SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_kg
FROM SPACEXTABLE
WHERE Customer = 'NASA (CRS)'
"""
pd.read_sql(query, con)

,Total_Payload_Mass_kg
0,45596


### Task 4: Display the average payload mass carried by booster version F9 v1.1

In [7]:
query = """
SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_Mass_kg
FROM SPACEXTABLE
WHERE Booster_Version = 'F9 v1.1'
"""
pd.read_sql(query, con)

,Avg_Payload_Mass_kg
0,2928.4


### Task 5: List the date when the first successful landing outcome on a ground pad was achieved

In [8]:
query = """
SELECT MIN(Date) AS First_Successful_Ground_Pad_Landing
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (ground pad)'
"""
pd.read_sql(query, con)

,First_Successful_Ground_Pad_Landing
0,2015-12-22


### Task 6: List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000

In [9]:
query = """
SELECT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ > 4000
  AND PAYLOAD_MASS__KG_ < 6000
"""
pd.read_sql(query, con)

,Booster_Version,PAYLOAD_MASS__KG_
0,F9 FT B1022,4696
1,F9 FT B1026,4600
2,F9 FT B1021.2,5300
3,F9 FT B1031.2,5200


### Task 7: List the total number of successful and failure mission outcomes

In [10]:
query = """
SELECT Mission_Outcome, COUNT(*) AS Total
FROM SPACEXTABLE
GROUP BY Mission_Outcome
"""
pd.read_sql(query, con)

,Mission_Outcome,Total
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


### Task 8: List the names of the booster versions which have carried the maximum payload mass

In [11]:
query = """
SELECT DISTINCT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTABLE
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE)
"""
pd.read_sql(query, con)

,Booster_Version,PAYLOAD_MASS__KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1049.4,15600
2,F9 B5 B1051.3,15600
3,F9 B5 B1056.4,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1051.4,15600
6,F9 B5 B1049.5,15600
7,F9 B5 B1060.2,15600
8,F9 B5 B1058.3,15600
9,F9 B5 B1051.6,15600


### Task 9: List the month names, failure landing outcomes in drone ship, booster versions and launch site for the months in year 2015

In [12]:
query = """
SELECT CASE substr(Date, 6, 2)
           WHEN '01' THEN 'January' WHEN '02' THEN 'February' WHEN '03' THEN 'March'
           WHEN '04' THEN 'April' WHEN '05' THEN 'May' WHEN '06' THEN 'June'
           WHEN '07' THEN 'July' WHEN '08' THEN 'August' WHEN '09' THEN 'September'
           WHEN '10' THEN 'October' WHEN '11' THEN 'November' WHEN '12' THEN 'December'
       END AS Month_Name,
       Landing_Outcome, Booster_Version, Launch_Site
FROM SPACEXTABLE
WHERE substr(Date, 1, 4) = '2015'
  AND Landing_Outcome = 'Failure (drone ship)'
"""
pd.read_sql(query, con)

,Month_Name,Landing_Outcome,Booster_Version,Launch_Site
0,January,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,April,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Task 10: Rank the count of landing outcomes between 2010-06-04 and 2017-03-20, in descending order

In [13]:
query = """
SELECT Landing_Outcome, COUNT(*) AS Outcome_Count,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Outcome_Count DESC
"""
pd.read_sql(query, con)

,Landing_Outcome,Outcome_Count,Rank
0,No attempt,10,1
1,Success (drone ship),5,2
2,Failure (drone ship),5,2
3,Success (ground pad),3,4
4,Controlled (ocean),3,4
5,Uncontrolled (ocean),2,6
6,Failure (parachute),2,6
7,Precluded (drone ship),1,8


In [14]:
con.close()